# Risk Analysis of Legal Clauses

## Objective
Identify and score potentially risky clauses in legal contracts.

In this notebook:
- We define what "risk" means in legal contracts
- Use keyword-based + ML-informed scoring
- Assign a risk score to each clause
`

In [1]:
import pandas as pd
import numpy as np
import pickle


In [2]:
# Load clause-level dataset
df = pd.read_csv("../data/processed/cuad_segmented_clauses.csv")

df.head()


,file_name,clause,pages,class_id,label,start_at,end_at,clean_text,sentences,word_count,clause_text
0,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,In the event that Licensor grants to another V...,2,8,Most Favored Nation,2558,2929,in the event that licensor grants to another v...,['in the event that licensor grants to another...,60,in the event that licensor grants to another v...
1,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,"If Licensor enters, or has entered, into an ag...",8,8,Most Favored Nation,18515,19562,if licensor enters or has entered into an agre...,['if licensor enters or has entered into an ag...,169,if licensor enters or has entered into an agre...
2,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,"Licensor shall provide to Rogers, no later tha...",8,8,Most Favored Nation,19563,20059,licensor shall provide to rogers no later than...,['licensor shall provide to rogers no later th...,79,licensor shall provide to rogers no later than...
3,IntegrityMediaInc_20010329_10-K405_EX-10.17_23...,"If for any reason, Integrity and TL are subjec...",3,8,Most Favored Nation,8173,8345,if for any reason integrity and tl are subject...,['if for any reason integrity and tl are subje...,30,if for any reason integrity and tl are subject...
4,TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4...,"The Company will, and Online BVI will cause th...",10,8,Most Favored Nation,30765,31577,the company will and online bvi will cause the...,['the company will and online bvi will cause t...,123,the company will and online bvi will cause the...


In [3]:
RISK_KEYWORDS = {
    "high": [
        "penalty", "terminate immediately", "without notice",
        "unlimited liability", "sole discretion",
        "indemnify", "in perpetuity"
    ],
    "medium": [
        "terminate", "liability", "damages",
        "breach", "governing law"
    ],
    "low": [
        "notice", "agreement", "confidential",
        "term", "obligations"
    ]
}


In [4]:
def calculate_risk_score(text):
    """
    Calculates risk score based on keyword presence.
    """
    score = 0
    text = text.lower()

    for word in RISK_KEYWORDS["high"]:
        if word in text:
            score += 3

    for word in RISK_KEYWORDS["medium"]:
        if word in text:
            score += 2

    for word in RISK_KEYWORDS["low"]:
        if word in text:
            score += 1

    return score


In [5]:
df["risk_score"] = df["clause_text"].apply(calculate_risk_score)

df[["clause_text", "label", "risk_score"]].head()


,clause_text,label,risk_score
0,in the event that licensor grants to another v...,Most Favored Nation,1
1,if licensor enters or has entered into an agre...,Most Favored Nation,3
2,licensor shall provide to rogers no later than...,Most Favored Nation,2
3,if for any reason integrity and tl are subject...,Most Favored Nation,0
4,the company will and online bvi will cause the...,Most Favored Nation,1


In [6]:
def risk_level(score):
    if score >= 6:
        return "High Risk"
    elif score >= 3:
        return "Medium Risk"
    else:
        return "Low Risk"

df["risk_level"] = df["risk_score"].apply(risk_level)

df["risk_level"].value_counts()


risk_level
Low Risk       7379
Medium Risk    1948
High Risk       371
Name: count, dtype: int64

In [7]:
# Load trained clause classifier
with open("../models/clause_classifier/tfidf.pkl", "rb") as f:
    tfidf = pickle.load(f)

with open("../models/clause_classifier/model.pkl", "rb") as f:
    classifier = pickle.load(f)


In [8]:
df["predicted_clause_type"] = classifier.predict(
    tfidf.transform(df["clause_text"])
)

df[["clause_text", "predicted_clause_type", "risk_level"]].head()


,clause_text,predicted_clause_type,risk_level
0,in the event that licensor grants to another v...,License Grant,Low Risk
1,if licensor enters or has entered into an agre...,License Grant,Medium Risk
2,licensor shall provide to rogers no later than...,Audit Rights,Low Risk
3,if for any reason integrity and tl are subject...,License Grant,Low Risk
4,the company will and online bvi will cause the...,License Grant,Low Risk


In [9]:
high_risk_clauses = df[df["risk_level"] == "High Risk"]

high_risk_clauses[["clause_text", "predicted_clause_type", "risk_score"]].head(5)


,clause_text,predicted_clause_type,risk_score
44,exxonmobil may terminate this agreement upon f...,Anti-Assignment,9
62,if following location acceptance of all cell s...,Cap On Liability,10
78,before expiration of the term either party may...,Anti-Assignment,10
89,in the event that the other party concludes in...,Anti-Assignment,9
93,this agreement shall terminate automatically w...,Anti-Assignment,8


In [10]:
import os

os.makedirs("../outputs", exist_ok=True)

df.to_csv("../outputs/clauses_with_risk_scores.csv", index=False)

print("Risk analysis results saved to outputs/clauses_with_risk_scores.csv")


Risk analysis results saved to outputs/clauses_with_risk_scores.csv


## Summary

In this notebook, we:
- Defined legal risk using keyword-based heuristics
- Assigned risk scores and risk levels to clauses
- Integrated clause classification with risk analysis
- Identified high-risk clauses for decision support

Next notebook:
➡ 05_simplification.ipynb
`